# M6 `e_long_song` — łączny czas nagrań WAV

Zliczamy wszystkie pliki `.wav` w:
- `ai/e_long_song`
- `human/e_long_song`

In [ ]:
from pathlib import Path

import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

PATHS = {
    "ai": Path("/net/people/plgrid/plgjedrzejkusnierz/scratch/data/M6/M6_database/ai/e_long_song"),
    "human": Path("/net/people/plgrid/plgjedrzejkusnierz/scratch/data/M6/M6_database/human/e_long_song"),
}

In [ ]:
def scan_wavs(root: Path, label: str) -> list[dict]:
    if not root.exists():
        raise FileNotFoundError(root)

    rows = []
    wav_files = sorted(root.glob("*.wav"))
    for wav_path in tqdm(wav_files, desc=label):
        info = sf.info(str(wav_path))
        rows.append({
            "label": label,
            "path": str(wav_path),
            "filename": wav_path.name,
            "duration_sec": info.duration,
            "sample_rate": info.samplerate,
            "channels": info.channels,
        })
    return rows


rows = []
for label, root in PATHS.items():
    rows.extend(scan_wavs(root, label))

df = pd.DataFrame(rows)
df["duration_min"] = df["duration_sec"] / 60
df["duration_h"] = df["duration_sec"] / 3600
df.head()

In [ ]:
summary = (
    df.groupby("label", as_index=False)
    .agg(
        n_files=("filename", "count"),
        total_sec=("duration_sec", "sum"),
        mean_sec=("duration_sec", "mean"),
        min_sec=("duration_sec", "min"),
        max_sec=("duration_sec", "max"),
    )
)
summary["total_hours"] = summary["total_sec"] / 3600
summary["total_min"] = summary["total_sec"] / 60

total_sec = df["duration_sec"].sum()
total_hours = total_sec / 3600
total_min = total_sec / 60

print("=== Per klasa ===")
display(summary)

print("\n=== RAZEM ===")
print(f"Pliki WAV:     {len(df)}")
print(f"Czas łącznie:  {total_sec:,.1f} s")
print(f"               {total_min:,.1f} min")
print(f"               {total_hours:,.2f} h  (~{int(total_hours)}h {int((total_hours % 1) * 60)}min)")